#Test
prueba de lectura de los reporrtes en formato html 

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from configs.config import RAW_HTML_DIR, JSON_REPORTS_DIR, PROJECT_ROOT

# --- Papermill parameters (overwritten at runtime) ---
input_path      = str(RAW_HTML_DIR)
output_dir      = str(JSON_REPORTS_DIR)
experiment_name = "baseline"
run_id          = "run_1"

In [ ]:
# Incluir esto al comienzo del notebook (después de la celda de parámetros si usas papermill)
import mlflow

# Asegurar que estamos en el run correcto sin iniciar uno nuevo
if mlflow.active_run() is None and "run_id" in globals():
    mlflow.start_run(run_id=run_id)


In [4]:
import os
import json
from bs4 import BeautifulSoup
import re
from typing import List, Dict

def split_html_into_pages(html: str) -> List[Dict[str, str]]:
    pattern = re.compile(
        r'(<div[^>]*?text-align:center[^>]*?>\s*'
        r'<div[^>]*?font-size:10pt[^>]*?>\s*'
        r'<font[^>]*?>\s*(\d+)\s*</font>\s*</div>\s*</div>)',
        re.IGNORECASE | re.DOTALL
    )
    matches = list(pattern.finditer(html))
    pages = []
    for i in range(len(matches)):
        start = matches[i].end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(html)
        page_number = int(matches[i].group(2))
        page_html = html[matches[i].start():end]
        pages.append({"page": page_number, "html": page_html})
    return pages

def extract_text_from_page(soup: BeautifulSoup) -> str:
    texts = soup.find_all("p")
    visible_text = []
    for p in texts:
        stripped = " ".join(p.stripped_strings)
        if stripped:
            visible_text.append(stripped)
    return "\n".join(visible_text)

def extract_tables_from_soup(soup: BeautifulSoup) -> List[Dict]:
    tables = soup.find_all("table")
    extracted_tables = []
    for table in tables:
        rows = table.find_all("tr")
        if not rows:
            continue
        parsed_table = []
        for row in rows:
            cells = row.find_all(["td", "th"])
            row_data = [" ".join(cell.stripped_strings) for cell in cells]
            if any(cell.strip() for cell in row_data):
                parsed_table.append(row_data)
        if len(parsed_table) >= 2:
            extracted_tables.append({
                "headers": parsed_table[0],
                "rows": parsed_table[1:]
            })
    return extracted_tables

def extract_text_and_tables_from_all_pages(html: str) -> List[Dict]:
    pages = split_html_into_pages(html)
    results = []
    for page in pages:
        soup = BeautifulSoup(page["html"], "html.parser")
        text = extract_text_from_page(soup)
        tables = extract_tables_from_soup(soup)
        results.append({
            "page": page["page"],
            "text": text,
            "tables": tables
        })
    return results

def save_full_result_to_json(result, output_path: str):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"✅ Guardado exitosamente en: {output_path}")


# === Lógica de procesamiento por lotes ===


os.makedirs(output_dir, exist_ok=True)

for filename in os.listdir(input_dir):
    if not filename.lower().endswith((".htm", ".html")):
        continue

    file_path = os.path.join(input_dir, filename)
    output_path = os.path.join(
        output_dir,
        f"{os.path.splitext(filename)[0]}_by_page.json"
    )

    print(f"\n📄 Procesando: {filename}")

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    result = extract_text_and_tables_from_all_pages(html)

    if not result:
        print("⚠️  No se detectaron páginas. Se guardará como página única.")
        soup = BeautifulSoup(html, "html.parser")
        text = extract_text_from_page(soup)
        tables = extract_tables_from_soup(soup)
        result = [{"page": 1, "text": text, "tables": tables}]

    total_tables = sum(len(page["tables"]) for page in result)
    print(f"📊 Tablas encontradas: {total_tables}")

    save_full_result_to_json(result, output_path)



📄 Procesando: a10-kfilingsquareinc2016.htm
⚠️  No se detectaron páginas. Se guardará como página única.
📊 Tablas encontradas: 80
✅ Guardado exitosamente en: E:\RAG_Project\data\processed\json_reports\a10-kfilingsquareinc2016_by_page.json

📄 Procesando: a2017123110-k.htm
📊 Tablas encontradas: 120
✅ Guardado exitosamente en: E:\RAG_Project\data\processed\json_reports\a2017123110-k_by_page.json

📄 Procesando: a201812dec3110k.htm
⚠️  No se detectaron páginas. Se guardará como página única.
📊 Tablas encontradas: 126
✅ Guardado exitosamente en: E:\RAG_Project\data\processed\json_reports\a201812dec3110k_by_page.json

📄 Procesando: adbe-20221202.html
⚠️  No se detectaron páginas. Se guardará como página única.
📊 Tablas encontradas: 75
✅ Guardado exitosamente en: E:\RAG_Project\data\processed\json_reports\adbe-20221202_by_page.json

📄 Procesando: adbe10kfy15.htm
⚠️  No se detectaron páginas. Se guardará como página única.
📊 Tablas encontradas: 96
✅ Guardado exitosamente en: E:\RAG_Project\data